In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcryff.trainer import ThermodynamicTrainer
from imolcryff.trainer.dmff_utils import parser_dmffyaml
from imolcryff.trainer.loss import loss_thermodynamicperturbation
from functools import partial

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/mdtraj/geometry/order.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/srak/apl/iMolCRAFT/DMFF/dmff/admp/qeq.py:33: UserWarning: jaxopt not found, QEQ cannot be used.
  warnings.warn("jaxopt not found, QEQ cannot be used.")
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="jsdivergence")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charges"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [6]:
from imolcryff.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[10].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[11].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[12].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[13].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[14].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[15].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[16].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[17].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[18].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9]], target_charges=[0.0])

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5179146628579856,0,--
2000,1.4975529894678976,141,1:12
3000,1.4897000793144528,132,1:16
4000,1.498564061039412,130,1:17
5000,1.4998195435071262,128,1:17
6000,1.4930652998803464,130,1:16
7000,1.496390299215196,129,1:15
8000,1.495737742542498,129,1:14
9000,1.4974651737586129,130,1:13
10000,1.4956723651984238,130,1:13
11000,1.4951404770545187,130,1:12
12000,1.4942314157738705,129,1:12
13000,1.4973566053082474,128,1:12
14000,1.493268642918712,128,1:11
15000,1.4958361690783042,128,1:11
16000,1.495505416811575,128,1:10
17000,1.4887188160956049,128,1:09
18000,1.4983203939201106,129,1:08
19000,1.4934287248692788,129,1:07
20000,1.4960513368002202,130,1:06
== Start Production ==
21000,1.4913521500166755,131,1:05
22000,1.4875492587234969,132,1:04
23000,1.4960489980326972,133,1:02
24000,1.4925722821254723,134,1:01
25000,1.4924418759

100%|██████████| 100/100 [00:02<00:00, 33.83it/s]


In [9]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)

Initial neff: [30]


In [10]:
trainer.neff = [50]

In [11]:
trainer.fit(10, 2)

100%|██████████| 100/100 [00:21<00:00,  4.74it/s]


Effective sample sizes:
  sample_0: 60
  Total: 61.027556015577865
Epoch 0 completed in 110.65 seconds.
Loss:  113.94264503529136
Best Loss:  113.94264503529136 at epoch  0


100%|██████████| 100/100 [00:21<00:00,  4.72it/s]


Effective sample sizes:
  sample_0: 61
  Total: 62.18479588971533
Epoch 1 completed in 83.60 seconds.
Loss:  113.80390546120938
Best Loss:  113.80390546120938 at epoch  1


100%|██████████| 100/100 [00:24<00:00,  4.08it/s]


...

Effective sample sizes:
  sample_0: 65
  Total: 66.00321609660219
Epoch 8 completed in 82.36 seconds.
Loss:  112.88830980262806
Best Loss:  112.88830980262806 at epoch  8


100%|██████████| 100/100 [00:21<00:00,  4.63it/s]


Effective sample sizes:
  sample_0: 64
  Total: 65.82006498439372
Epoch 9 completed in 74.39 seconds.
Loss:  112.76545360313095
Best Loss:  112.76545360313095 at epoch  9


100%|██████████| 100/100 [00:21<00:00,  4.74it/s]


Effective sample sizes:
  sample_0: 64
  Total: 65.44127410366102
Epoch 10 completed in 80.96 seconds.
Loss:  112.64446307690751
Best Loss:  112.64446307690751 at epoch  10


In [12]:
trainer.neff = neff_prev

In [13]:
trainer.fit(490, 2)

100%|██████████| 100/100 [00:21<00:00,  4.67it/s]


Effective sample sizes:
  sample_0: 63
  Total: 64.86779933820006
Epoch 11 completed in 74.11 seconds.
Loss:  112.52526308376741
Best Loss:  112.52526308376741 at epoch  11


100%|██████████| 100/100 [00:21<00:00,  4.61it/s]


Effective sample sizes:
  sample_0: 63
  Total: 64.10257553817854
Epoch 12 completed in 77.78 seconds.
Loss:  112.40776971084148
Best Loss:  112.40776971084148 at epoch  12


100%|██████████| 100/100 [00:21<00:00,  4.63it/s]


...

Epoch 281 completed in 179.63 seconds.
Loss:  100.63349432985761
Best Loss:  99.97495961324648 at epoch  271


100%|██████████| 100/100 [00:23<00:00,  4.33it/s]


Resampling sample_0... by xmlfiles/epoch-282.xml
Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5086225164824092,0,--
2000,1.5033292420926911,175,0:58
3000,1.5033868683047038,169,0:59
4000,1.4996016268406305,170,0:58
5000,1.504583101594092,172,0:57
6000,1.4964155617709145,171,0:57
7000,1.4991653601452337,172,0:56
8000,1.5041704541084455,172,0:56
9000,1.5001740973401418,173,0:55
10000,1.4959078359049127,173,0:54
11000,1.497792231063156,174,0:54
12000,1.4977740201574854,175,0:53
13000,1.4974491567357096,176,0:52
14000,1.50355544044291,176,0:51
15000,1.5014573151723518,177,0:51
16000,1.4995585586015319,177,0:50
17000,1.49750894354684,177,0:50
18000,1.4959170254016727,178,0:49
19000,1.4968722794549423,178,0:49
20000,1.4976841784800674,178,0:48
== Start Production ==
21000,1.4999386146014453,178,0:48
22000,1.5034640275131521,178,0:47
23000,1.4930828916650107,178,0:47
2400

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 100/100 [00:01<00:00, 53.26it/s]


ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape float64[]
The error occurred in the __reduce__ method, which may indicate an attempt to serialize/pickle a traced value.

See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.ConcretizationTypeError

In [ ]:
trainer.fit(40, 2)

In [ ]:
# trainer.from_checkpoint(trainer_checkpoint="train_state.pkl",
#                         ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
#                         nums_ffxml=[2,1,1],
#                         pdbfile="merged_supercell_bonds.pdb",
#                         initial_ffxml="xmlfiles/epoch-4.xml",
#                         loss_fn=lossfn,
#                         opt_fftypes=["NonbondedForce/charges"],
#                         )